In [1]:
import os
import sys
import zipfile
import requests
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split


In [8]:
# ====================== DOWNLOAD MOVIELENS 1M ======================
def download_movielens():
    url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
    zip_path = "ml-1m.zip"
    extract_path = "ml-1m"
    
    if not os.path.exists(extract_path):
        print("Downloading MovieLens 1M dataset...")
        r = requests.get(url, stream=True)
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        
        print("Extracting...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(".")
        os.remove(zip_path)
        print("Download complete!")
    else:
        print("MovieLens 1M already downloaded.")

download_movielens()



MovieLens 1M already downloaded.


In [2]:
# ====================== LOAD & FILTER ======================
ratings = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)

# Keep only positive feedback (standard for Mult-VAE)
ratings = ratings[ratings["rating"] >= 3].copy()

# Users with at least 20 positive ratings
user_counts = ratings.groupby("userId").size()
valid_users = user_counts[user_counts >= 20].index.tolist()

print(f"Users with ≥20 ratings (rating ≥3): {len(valid_users)}")

# ====================== PER-USER INPUT / OUTPUT (temporal last-10-out) ======================
print("Building per-user input/output movies (this may take a few seconds)...")
user_data = {}          # uid → {'input': set of movieIds, 'output': set of movieIds}

grouped = ratings.groupby("userId")
for uid in valid_users:
    user_df = grouped.get_group(uid).sort_values("timestamp")
    movies = user_df["movieId"].tolist()          # already sorted by time
    
    if len(movies) >= 20:
        input_movies = movies[:-10]               # all except last 10
        output_movies = movies[-10:]              # only the last 10
        
        user_data[uid] = {
            "input": set(input_movies),           # set for fast lookup
            "output": set(output_movies)
        }

print(f"Final users after last-10-out filtering: {len(user_data)}")

# ====================== GLOBAL MOVIE MAPPING ======================
all_movies = set()
for d in user_data.values():
    all_movies.update(d["input"])
    all_movies.update(d["output"])

movie_ids = sorted(all_movies)
movie_map = {mid: idx for idx, mid in enumerate(movie_ids)}
n_items = len(movie_ids)

print(f"Total items: {n_items}")

# ====================== SPLIT USERS (80% train / 10% val / 10% test) ======================
user_ids = list(user_data.keys())
np.random.seed(42)                    # reproducible split
np.random.shuffle(user_ids)

n_users = len(user_ids)
n_train = int(0.8 * n_users)
n_val   = int(0.1 * n_users)
n_test  = n_users - n_train - n_val

train_users = user_ids[:n_train]
val_users   = user_ids[n_train:n_train + n_val]
test_users  = user_ids[n_train + n_val:]

print(f"Train users: {len(train_users)} | Val users: {len(val_users)} | Test users: {len(test_users)}")


# ====================== BUILD MATRICES ======================
def build_matrices(user_list, user_data, movie_map, n_items):
    n = len(user_list)
    input_mat  = np.zeros((n, n_items), dtype=np.float32)
    output_mat = np.zeros((n, n_items), dtype=np.float32)
    
    user_map_split = {uid: idx for idx, uid in enumerate(user_list)}
    
    for uid in user_list:
        idx = user_map_split[uid]
        # Input matrix
        for mid in user_data[uid]["input"]:
            if mid in movie_map:
                input_mat[idx, movie_map[mid]] = 1.0
        # Output matrix (only last 10)
        for mid in user_data[uid]["output"]:
            if mid in movie_map:
                output_mat[idx, movie_map[mid]] = 1.0
    return input_mat, output_mat

print("Building train matrices...")
train_matrix_i, train_matrix_o = build_matrices(train_users, user_data, movie_map, n_items)

print("Building validation matrices...")
valid_matrix_i,   valid_matrix_o   = build_matrices(val_users,   user_data, movie_map, n_items)

print("Building test matrices...")
test_matrix_i,  test_matrix_o  = build_matrices(test_users,  user_data, movie_map, n_items)

print("\nAll matrices ready!")
print(f"Shape example → train_input: {train_matrix_i.shape} | train_output: {train_matrix_o.shape}")

Users with ≥20 ratings (rating ≥3): 5755
Building per-user input/output movies (this may take a few seconds)...
Final users after last-10-out filtering: 5755
Total items: 3624
Train users: 4604 | Val users: 575 | Test users: 576
Building train matrices...
Building validation matrices...
Building test matrices...

All matrices ready!
Shape example → train_input: (4604, 3624) | train_output: (4604, 3624)


In [3]:
# ====================== DATASET ======================
class UserDataset(Dataset):
    def __init__(self, input_matrix, output_matrix):
        self.input_data  = torch.from_numpy(input_matrix).float()
        self.output_data = torch.from_numpy(output_matrix).float()
   
    def __len__(self):
        return len(self.input_data)
   
    def __getitem__(self, idx):
        return self.input_data[idx], self.output_data[idx]   # returns (input, target)

#train_dataset = UserDataset(train_matrix)
#train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2)

train_dataset = UserDataset(train_matrix_i, train_matrix_o)
valid_dataset   = UserDataset(valid_matrix_i,   valid_matrix_o)
test_dataset  = UserDataset(test_matrix_i,  test_matrix_o)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True,  num_workers=2)
valid_loader = DataLoader(valid_dataset,   batch_size=512, shuffle=False, num_workers=2)

In [4]:
# ====================== MULT-VAE MODEL ======================
class MultVAE(nn.Module):
    def __init__(self, n_items, hidden_dim=400, latent_dim=100, dropout=0.5):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_items, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout)
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_items)
        )
    
    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultVAE(n_items).to(device)
optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=0.0)



In [5]:
# ====================== LOSS (Multinomial + KL) ======================
def mult_vae_loss(logits, target, mu, logvar, anneal=1.0):
    # Reconstruction: negative multinomial log-likelihood
    log_softmax = torch.log_softmax(logits, dim=1)
    recon_loss = -torch.sum(target * log_softmax, dim=1)
    
    # KL divergence (standard Gaussian prior)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    
    return torch.mean(recon_loss + anneal * kl_loss)



In [9]:
class StudentMultVAE(nn.Module):
    def __init__(self, n_items, hidden_dim=200, latent_dim=50, dropout=0.5):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_items, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout)
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
       
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_items)
        )
   
    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
   
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
   
    def decode(self, z):
        return self.decoder(z)
   
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar

def minillm_distillation_loss(student_logits, teacher_logits, target,
                              temperature=2.5, alpha=0.65, beta=0.35):
    """
    MiniLLM-style loss adapted for MultiVAE:
    - Heavy emphasis on Reverse KL (student || teacher) on the output distribution
    - No latent KL when dimensions differ
    - Keeps hard reconstruction for recommendation quality
    """
    # Temperature scaling for softer distributions
    student_probs = torch.softmax(student_logits / temperature, dim=1)
    teacher_probs = torch.softmax(teacher_logits / temperature, dim=1)
    
    # === Core: Reverse KL (student || teacher) - this is the spirit of MiniLLM ===
    # Encourages student to focus on teacher's high-probability modes
    reverse_kl = torch.sum(
        student_probs * (torch.log(student_probs + 1e-9) - torch.log(teacher_probs + 1e-9)),
        dim=1
    ).mean()
    
    # === Small forward KL for stability (optional but helps training) ===
    forward_kl = torch.sum(
        teacher_probs * (torch.log(teacher_probs + 1e-9) - torch.log(student_probs + 1e-9)),
        dim=1
    ).mean()
    
    # === Hard reconstruction loss (important for MultiVAE to stay useful) ===
    log_softmax_student = torch.log_softmax(student_logits, dim=1)
    hard_recon_loss = -torch.sum(target * log_softmax_student, dim=1).mean()
    
    # Total loss
    loss = alpha * reverse_kl + 0.15 * forward_kl + beta * hard_recon_loss
    
    return loss



In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load teacher (frozen)
teacher = MultVAE(n_items, hidden_dim=400, latent_dim=100, dropout=0.5).to(device)
teacher.load_state_dict(torch.load("multvae_movielens.pth", map_location=device))
teacher.eval()

# Smaller student
student = StudentMultVAE(n_items, hidden_dim=200, latent_dim=50, dropout=0.5).to(device)

optimizer = optim.Adam(student.parameters(), lr=1e-4, weight_decay=1e-5)

epochs = 60
alpha = 0.65      # Higher weight on reverse KL (MiniLLM flavor)
beta = 0.35       # Weight on hard reconstruction
temperature = 2.5

best_val_loss = float('inf')

print("Starting MiniLLM-style distillation (Reverse KL on outputs)...")

for epoch in range(epochs):
    student.train()
    total_loss = 0.0
    
    for input_batch, output_batch in train_loader:
        input_batch = input_batch.to(device)
        output_batch = output_batch.to(device)
        
        optimizer.zero_grad()
        
        with torch.no_grad():
            teacher_logits, _, _ = teacher(input_batch)   # We only need logits from teacher
        
        student_logits, _, _ = student(input_batch)
        
        loss = minillm_distillation_loss(
            student_logits, teacher_logits, output_batch,
            temperature=temperature, alpha=alpha, beta=beta
        )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 5.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_train_loss = total_loss / len(train_loader)
    
    # Validation (using original loss for monitoring)
    student.eval()
    val_loss = 0.0
    with torch.no_grad():
        for input_batch, output_batch in valid_loader:
            input_batch = input_batch.to(device)
            output_batch = output_batch.to(device)
            s_logits, _, _ = student(input_batch)
            # Use your original loss function (mu/logvar ignored for simplicity)
            loss = mult_vae_loss(s_logits, output_batch, torch.zeros_like(s_logits), torch.zeros_like(s_logits))
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(valid_loader)
    
    print(f"Epoch {epoch+1:2d}/{epochs} | Train Distill Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(student.state_dict(), "student_multvae_minillm_fixed.pth")

print("Distillation finished! Student saved.")

Starting MiniLLM-style distillation (Reverse KL on outputs)...
Epoch  1/60 | Train Distill Loss: 29.0614 | Val Loss: 82.2798
Epoch  2/60 | Train Distill Loss: 29.0400 | Val Loss: 82.1526
Epoch  3/60 | Train Distill Loss: 29.0032 | Val Loss: 82.0778
Epoch  4/60 | Train Distill Loss: 28.9889 | Val Loss: 82.0441
Epoch  5/60 | Train Distill Loss: 28.9336 | Val Loss: 81.9750
Epoch  6/60 | Train Distill Loss: 28.8889 | Val Loss: 81.7733
Epoch  7/60 | Train Distill Loss: 28.8210 | Val Loss: 81.4759
Epoch  8/60 | Train Distill Loss: 28.7191 | Val Loss: 81.0791
Epoch  9/60 | Train Distill Loss: 28.5704 | Val Loss: 80.5189
Epoch 10/60 | Train Distill Loss: 28.3401 | Val Loss: 79.8856
Epoch 11/60 | Train Distill Loss: 28.0709 | Val Loss: 79.0554
Epoch 12/60 | Train Distill Loss: 27.7640 | Val Loss: 78.1329
Epoch 13/60 | Train Distill Loss: 27.4631 | Val Loss: 77.3137
Epoch 14/60 | Train Distill Loss: 27.1718 | Val Loss: 76.5954
Epoch 15/60 | Train Distill Loss: 26.9548 | Val Loss: 76.0280
Epoch 1

In [12]:

model = student

# ====================== RECOMMENDATION FUNCTION ======================
def recommend_for_user(model, user_idx, train_interactions, k=10):
    model.eval()
    with torch.no_grad():
        mean_score = []
        for i in range(10):
            user_vec = torch.tensor(train_interactions[user_idx]).unsqueeze(0).float().to(device)
            logits, _, _ = model(user_vec)
            scores = logits.squeeze().cpu().numpy()
            if i == 0:
                mean_score = scores/10
            else:
                mean_score += scores/10
        
        # Mask already seen items
        seen = train_interactions[user_idx] > 0
        mean_score[seen] = -np.inf
        
        top_k_idx = np.argsort(scores)[-k:][::-1]
        return top_k_idx

# Load movie titles for nice output
movies = pd.read_csv(
    "ml-1m/movies.dat",
    sep="::",
    engine="python",
    names=["movieId", "title", "genres"],
    encoding="ISO-8859-1"
)
movie_title_map = {}
for _, row in movies.iterrows():
    if row["movieId"] in movie_map:
        movie_title_map[movie_map[row["movieId"]]] = row["title"]

# Example recommendation for user 0 (change user_idx as you like)
user_idx = 0
top_movies_idx = recommend_for_user(model, user_idx, test_matrix_i, k=10)
print("\n=== Recommended movies for user", test_users[user_idx], "===")
for rank, item_idx in enumerate(top_movies_idx, 1):
    title = movie_title_map.get(item_idx, f"Unknown (ID {item_idx})")
    print(f"{rank}. {title}")




=== Recommended movies for user 3471 ===
1. Gladiator (2000)
2. Sixth Sense, The (1999)
3. Shakespeare in Love (1998)
4. X-Men (2000)
5. American Beauty (1999)
6. Who Framed Roger Rabbit? (1988)
7. Toy Story 2 (1999)
8. Men in Black (1997)
9. Talented Mr. Ripley, The (1999)
10. High Fidelity (2000)


In [13]:
def elliotize_data(recs_dict, recs_tsv_path):
  data = []
  for user_id, rec_list in recs_dict.items():
      for rec in rec_list:
          item_id = rec["item"]
          score = float(rec["score"])
          data.append([str(user_id), str(item_id), score])

  # Create DataFrame and ensure sorting per user (score descending)
  df_recs = pd.DataFrame(data, columns=["user", "item", "score"])
  df_recs = df_recs.sort_values(by=["user", "score"], ascending=[True, False])

  #recs_tsv_path = "my_recommendations.tsv"
  df_recs.to_csv(recs_tsv_path, sep="\t", header=False, index=False)
  print(f"Recommendations saved to {recs_tsv_path} (format: user\titem\tscore)")

In [14]:
test_dict = {}
reverse_movie_map = {idx: mid for mid, idx in movie_map.items()}
reverse_user_map = user_map_split = {idx: uid for idx, uid in enumerate(test_users)}

for user_idx in range(len(test_matrix_i)):
    top_movies_idx = recommend_for_user(model, user_idx, test_matrix_i, k=10)
    movie_rank = []
    for m_i, movie in enumerate(top_movies_idx):
      if m_i < 10:
        movie_rank.append({
            "item": reverse_movie_map[movie],
            "score": 1
        })
    test_dict[f"user{reverse_user_map[user_idx]}"] = movie_rank


elliotize_data(test_dict, f"MultiVAE_elliot_tsv/MultiVAE_distilled_elliot_rec.tsv")

Recommendations saved to MultiVAE_elliot_tsv/MultiVAE_distilled_elliot_rec.tsv (format: user	item	score)


In [15]:
train_dict = {}
reverse_movie_map = {idx: mid for mid, idx in movie_map.items()}
reverse_user_map = user_map_split = {idx: uid for idx, uid in enumerate(train_users)}

for user_idx in range(len(train_matrix_o)):
    top_movies_idx = np.argsort(train_matrix_o[user_idx])[-10:][::-1]
    movie_rank = []
    for m_i, movie in enumerate(top_movies_idx):
      if m_i < 10:
        movie_rank.append({
            "item": reverse_movie_map[movie],
            "score": 1
        })
    train_dict[f"user{reverse_user_map[user_idx]}"] = movie_rank

elliotize_data(train_dict, f"MultiVAE_elliot_tsv/MultiVAE_elliot_train.tsv")

valid_dict = {}
reverse_movie_map = {idx: mid for mid, idx in movie_map.items()}
reverse_user_map = user_map_split = {idx: uid for idx, uid in enumerate(val_users)}

for user_idx in range(len(valid_matrix_o)):
    top_movies_idx = np.argsort(valid_matrix_o[user_idx])[-10:][::-1]
    movie_rank = []
    for m_i, movie in enumerate(top_movies_idx):
      if m_i < 10:
        movie_rank.append({
            "item": reverse_movie_map[movie],
            "score": 1
        })
    valid_dict[f"user{reverse_user_map[user_idx]}"] = movie_rank

elliotize_data(valid_dict, f"MultiVAE_elliot_tsv/MultiVAE_elliot_valid.tsv")

test_dict = {}
reverse_movie_map = {idx: mid for mid, idx in movie_map.items()}
reverse_user_map = user_map_split = {idx: uid for idx, uid in enumerate(test_users)}

for user_idx in range(len(test_matrix_o)):
    top_movies_idx = np.argsort(test_matrix_o[user_idx])[-10:][::-1]
    movie_rank = []
    for m_i, movie in enumerate(top_movies_idx):
      if m_i < 10:
        movie_rank.append({
            "item": reverse_movie_map[movie],
            "score": 1
        })
    test_dict[f"user{reverse_user_map[user_idx]}"] = movie_rank

elliotize_data(test_dict, f"MultiVAE_elliot_tsv/MultiVAE_elliot_test.tsv")

Recommendations saved to MultiVAE_elliot_tsv/MultiVAE_elliot_train.tsv (format: user	item	score)
Recommendations saved to MultiVAE_elliot_tsv/MultiVAE_elliot_valid.tsv (format: user	item	score)
Recommendations saved to MultiVAE_elliot_tsv/MultiVAE_elliot_test.tsv (format: user	item	score)


In [ ]:
print("Based on the user's watching history and the film's rating, order the list of candidate films. Order the films where the first one of the list is the most likely to be watched by the user. The output should only contain the ordered list of the recommended films bounded by the  special strings '%% START RECOMMENDED LIST %%' and '%% END LIST %%'. Here follows the user's history and the list of candidate films.\n%% START HISTORY %%\nMovie name: Toy Story 2 (1999)\tRating:4\nMovie name: Airplane! (1980)\tRating:4\nMovie name: Pleasantville (1998)\tRating:3\nMovie name: Dumbo (1941)\tRating:5\nMovie name: Princess Bride, The (1987)\tRating:3\nMovie name: Snow White and the Seven Dwarfs (1937)\tRating:4\nMovie name: Miracle on 34th Street (1947)\tRating:4\nMovie name: Ponette (1996)\tRating:4\nMovie name: Schindler's List (1993)\tRating:5\nMovie name: Aladdin (1992)\tRating:4\n%% END HISTORY %%\n%% START CANDIDATES %%\nMovie name: 13th Warrior, The (1999)\nMovie name: Dogma (1999)\nMovie name: Unforgiven (1992)\nMovie name: Dirty Dozen, The (1967)\nMovie name: Gladiator (2000)\nMovie name: Firm, The (1993)\nMovie name: Sling Blade (1996)\nMovie name: Fight Club (1999)\nMovie name: Simple Plan, A (1998)\nMovie name: Addams Family, The (1991)\nMovie name: Caddyshack (1980)\nMovie name: Star Trek III: The Search for Spock (1984)\nMovie name: Breakfast Club, The (1985)\nMovie name: Dark City (1998)\nMovie name: Jackie Brown (1997)\nMovie name: Shining, The (1980)\nMovie name: Terminator, The (1984)\nMovie name: Lost World: Jurassic Park, The (1997)\nMovie name: Blair Witch Project, The (1999)\nMovie name: Heat (1995)\nMovie name: Beauty and the Beast (1991)\nMovie name: Crocodile Dundee (1986)\nMovie name: What About Bob? (1991)\nMovie name: Rocketeer, The (1991)\nMovie name: Batman (1989)\nMovie name: When Harry Met Sally... (1989)\nMovie name: Blazing Saddles (1974)\nMovie name: Apocalypse Now (1979)\nMovie name: Air Force One (1997)\nMovie name: Star Trek: The Wrath of Khan (1982)\nMovie name: Edward Scissorhands (1990)\nMovie name: Meet the Parents (2000)\nMovie name: Deep Impact (1998)\nMovie name: Mad Max 2 (a.k.a. The Road Warrior) (1981)\nMovie name: Bull Durham (1988)\nMovie name: Star Trek IV: The Voyage Home (1986)\nMovie name: Witness (1985)\nMovie name: Goonies, The (1985)\nMovie name: Basic Instinct (1992)\nMovie name: Manchurian Candidate, The (1962)\nMovie name: Talented Mr. Ripley, The (1999)\nMovie name: As Good As It Gets (1997)\nMovie name: Mummy, The (1999)\nMovie name: True Lies (1994)\nMovie name: Robocop (1987)\nMovie name: Batman Returns (1992)\nMovie name: My Best Friend's Wedding (1997)\nMovie name: Parenthood (1989)\nMovie name: Clockwork Orange, A (1971)\nMovie name: Almost Famous (2000)\nMovie name: Scream (1996)\nMovie name: Repo Man (1984)\nMovie name: Wrong Trousers, The (1993)\nMovie name: Hercules (1997)\nMovie name: Gremlins (1984)\nMovie name: Good Will Hunting (1997)\nMovie name: Citizen Kane (1941)\nMovie name: League of Their Own, A (1992)\nMovie name: Toy Story (1995)\nMovie name: Payback (1999)\nMovie name: Amadeus (1984)\nMovie name: Player, The (1992)\nMovie name: Shakespeare in Love (1998)\nMovie name: Grease (1978)\nMovie name: Hunt for Red October, The (1990)\nMovie name: Dog Day Afternoon (1975)\nMovie name: Tarzan (1999)\nMovie name: Back to the Future Part II (1989)\nMovie name: Pocahontas (1995)\nMovie name: Close Encounters of the Third Kind (1977)\nMovie name: Poltergeist (1982)\nMovie name: Fish Called Wanda, A (1988)\nMovie name: Dune (1984)\nMovie name: Splash (1984)\nMovie name: Hunchback of Notre Dame, The (1996)\nMovie name: Patriot, The (2000)\nMovie name: From Dusk Till Dawn (1996)\nMovie name: Big Chill, The (1983)\nMovie name: Fatal Attraction (1987)\nMovie name: Taxi Driver (1976)\nMovie name: Godfather: Part II, The (1974)\nMovie name: Alien (1979)\nMovie name: GoldenEye (1995)\nMovie name: Nightmare Before Christmas, The (1993)\nMovie name: Twister (1996)\nMovie name: Tomorrow Never Dies (1997)\nMovie name: Grifters, The (1990)\nMovie name: Antz (1998)\nMovie name: Cocoon (1985)\nMovie name: Entrapment (1999)\nMovie name: Bug's Life, A (1998)\nMovie name: Dave (1993)\nMovie name: L.A. Story (1991)\nMovie name: Little Shop of Horrors (1986)\nMovie name: Mulan (1998)\nMovie name: Go (1999)\nMovie name: Close Shave, A (1995)\nMovie name: You've Got Mail (1998)\nMovie name: Leaving Las Vegas (1995)\nMovie name: Star Trek: The Motion Picture (1979)\n%% END CANDIDATES %%\n\n\n%% START RECOMMENDED LIST %%\nMovie name: Pocahontas (1995)\nMovie name: Bug's Life, A (1998)\nMovie name: Beauty and the Beast (1991)\nMovie name: Toy Story (1995)\nMovie name: Hercules (1997)\nMovie name: Mulan (1998)\nMovie name: Antz (1998)\nMovie name: Hunchback of Notre Dame, The (1996)\nMovie name: Tarzan (1999)\nMovie name: Close Shave, A (1995)\nMovie name: Heat (1995)\nMovie name: GoldenEye (1995)\nMovie name: Leaving Las Vegas (1995)\nMovie name: From Dusk Till Dawn (1996)\nMovie name: Taxi Driver (1976)\nMovie name: True Lies (1994)\nMovie name: Dave (1993)\nMovie name: Firm, The (1993)\nMovie name: Nightmare Before Christmas, The (1993)\nMovie name: Batman (1989)\nMovie name: Alien (1979)\nMovie name: Citizen Kane (1941)\nMovie name: Hunchback of Notre Dame, The (1996)\nMovie name: Clockwork Orange, A (1971)\nMovie name: Scream (1996)\nMovie name: Twister (1996)\nMovie name: Basic Instinct (1992)\nMovie name: When Harry Met Sally... (1989)\nMovie name: Grifters, The (1990)\nMovie name: Fish Called Wanda, A (1988)\nMovie name: Wrong Trousers, The (1993)\nMovie name: Terminator, The (1984)\nMovie name: Shining, The (1980)\nMovie name: Unforgiven (1992)\nMovie name: Manchurian Candidate, The (1962)\nMovie name: Apocalypse Now (1979)\nMovie name: Blazing Saddles (1974)\nMovie name: Star Trek III: The Search for Spock (1984)\nMovie name: Firm, The (1993)\nMovie name: Basic Instinct (1992)\nMovie name: Batman Returns (1992)\nMovie name: Firm, The (1993)\nMovie name: As Good As It Gets (1997)\nMovie name: Clockwork Orange, A (1971)\nMovie name: Sling Blade (1996)\nMovie name: Wrong Trousers, The (1993)\nMovie name: Wrong Trousers, The (")


In [24]:
# ====================== SIMPLE EVALUATION (Recall@10 & NDCG@10) ======================
@torch.no_grad()
def evaluate(model, test_input, test_output, k=10):
    model.eval()
    recalls = []
    ndcgs = []
    
    for u in range(n_users):
        if np.sum(test_input[u]) == 0:
            continue
            
        user_vec = torch.tensor(test_input[u]).unsqueeze(0).float().to(device)
        logits = model(user_vec)[0].squeeze().cpu().numpy()
        
        # Mask train items
        scores = logits.copy()
        scores[test_input[u] > 0] = -np.inf
        
        ranked = np.argsort(scores)[::-1]
        true_items = set(np.where(test_output[u] > 0)[0])
        
        # Recall@K
        hits = len(set(ranked[:k]) & true_items)
        recall = hits / len(true_items)
        recalls.append(recall)
        
        # NDCG@K
        dcg = sum(1 / np.log2(i + 2) for i, item in enumerate(ranked[:k]) if item in true_items)
        idcg = sum(1 / np.log2(i + 2) for i in range(min(k, len(true_items))))
        ndcg = dcg / idcg if idcg > 0 else 0
        ndcgs.append(ndcg)
    
    return np.mean(recalls), np.mean(ndcgs)

rec, ndcg = evaluate(model, test_matrix_i, test_matrix_o, k=10)
print(f"\nEvaluation on held-out test set:")
print(f"Recall@10: {rec:.4f}")
print(f"NDCG@10 : {ndcg:.4f}")


Evaluation on held-out test set:
Recall@10: 0.0217
NDCG@10 : 0.0222
